# Voice access project - final notebook

This notebook is the lightweight interface notebook.  
It loads one selected model and predicts `allow` vs `not_allow` for a WAV file.

In [ ]:
from pathlib import Path
import pandas as pd
import torch

from voice_project_code import ProjectPaths, make_loaders, predict_wav_file

PATHS = ProjectPaths.from_work_dir("voice_project_artifacts")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BEST_MODEL_NAME = "smallcnn"
BEST_CHECKPOINT = PATHS.model_dir / "baseline_smallcnn_adam.pt"

IMAGE_SIZE = 128
BATCH_SIZE = 32
SEGMENT_SECONDS = 3.0
DROPOUT = 0.30
USE_BATCHNORM = True

_, _, _, loader_stats = make_loaders(
    paths=PATHS,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    normalize=True,
    augment=False,
)

print("DEVICE =", DEVICE)
print("BEST_CHECKPOINT =", BEST_CHECKPOINT)
print(loader_stats)

## Predict from an existing WAV file
Replace `WAV_PATH` with a real file path.

In [ ]:
WAV_PATH = Path(r"some_test_audio.wav")

result = predict_wav_file(
    wav_path=WAV_PATH,
    checkpoint_path=BEST_CHECKPOINT,
    model_name=BEST_MODEL_NAME,
    train_mean=loader_stats["train_mean"],
    train_std=loader_stats["train_std"],
    segment_seconds=SEGMENT_SECONDS,
    image_size=IMAGE_SIZE,
    device=DEVICE,
    dropout=DROPOUT,
    use_batchnorm=USE_BATCHNORM,
)

result

## Optional: microphone recording first, then prediction
Uncomment only if you want a simple notebook-based recording flow and `sounddevice` is installed.

In [ ]:
# import sounddevice as sd
# import soundfile as sf
# import numpy as np
#
# RECORD_SECONDS = 5
# SAMPLE_RATE = 16000
# OUT_WAV = Path("recorded_from_microphone.wav")
#
# audio = sd.rec(int(RECORD_SECONDS * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype="float32")
# sd.wait()
# audio = np.squeeze(audio, axis=1)
# sf.write(OUT_WAV, audio, SAMPLE_RATE)
#
# result = predict_wav_file(
#     wav_path=OUT_WAV,
#     checkpoint_path=BEST_CHECKPOINT,
#     model_name=BEST_MODEL_NAME,
#     train_mean=loader_stats["train_mean"],
#     train_std=loader_stats["train_std"],
#     segment_seconds=SEGMENT_SECONDS,
#     image_size=IMAGE_SIZE,
#     device=DEVICE,
#     dropout=DROPOUT,
#     use_batchnorm=USE_BATCHNORM,
# )
#
# result